# Ciringoni E2 evaluasi ulang di Kaggle CPU
Bandingkan checkpoint historis native 640 dan 1280 dari lima epoch dengan sumber dan split yang sama. Versi ini menjalankan prediksi baru, evaluasi pada 356 gambar validation, dan benchmark CPU empat thread. Pelatihan ulang E2 memiliki notebook terpisah.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.210', 'pycocotools==2.0.11'], check=True)

In [ ]:
import csv, json, tempfile, tarfile
from pathlib import Path
import torch
files = list(Path('/kaggle/input').rglob('validation_e2.tar.gz'))
assert len(files) <= 1, files
project = Path(tempfile.mkdtemp(prefix='ciringoni-e2-'))
if files:
    with tarfile.open(files[0]) as archive:
        archive.extractall(project, filter='data')
else:
    import shutil
    manifests = list(Path('/kaggle/input').rglob('geometry.json'))
    assert len(manifests) == 1, manifests
    shutil.copytree(manifests[0].parent, project, dirs_exist_ok=True)
gt = json.loads((project/'hasil_m1/coco_val_gt.json').read_text())
assert (len(gt['images']), len(gt['annotations'])) == (356, 2205)
import hashlib, os, platform
from importlib.metadata import version

affinity = sorted(os.sched_getaffinity(0))[:4]
assert len(affinity) == 4
os.sched_setaffinity(0, affinity)
os.environ.update(OMP_NUM_THREADS="4", MKL_NUM_THREADS="4")
device = "cpu"
hashes = {
    640: "56568daf2c930033058a3b0468f1357069b1110ce2a1e07e705405547f46d34f",
    1280: "0bd467aef478ebd26555074030ff653c52cf05ad5380fcbe66057b3c995f264e",
}
def digest(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

output = Path('/kaggle/working')
print('Validasi:', len(gt['images']), 'gambar;', len(gt['annotations']), 'kotak. Perangkat:', device)

## Gambar, prediksi dan skor
Kaggle menilai kedua checkpoint historis pada validation. CPU empat thread mengukur latensi median dan p95.

In [ ]:
rows = []
for size, run in [(640,'E2_control_native_640'), (1280,'E2_native_1280')]:
    data = project/str(size)
    yaml = data/'data.yaml'
    yaml.write_text(f'path: {data}\ntrain: images/val\nval: images/val\nnames: [infected]\n')
    dest = output/run
    command = [sys.executable, str(project/'hasil_m2/evaluasi_e2.py'), str(data/'best.pt'), str(yaml), str(project/'geometry.json'), str(dest), '--size', str(size), '--device', device]
    assert digest(data / "best.pt") == hashes[size]
    subprocess.run(command, check=True)
    subprocess.run(command + ["--cpu-benchmark"], check=True)
    row = json.loads((dest / "metrik.json").read_text())
    assert row["gambar"] == 356 and row["cpu_torch_threads"] == 4
    row["checkpoint"] = f"{size}/best.pt"
    row["checkpoint_sha256"] = hashes[size]
    rows.append(row)
    print(run, 'AP-small', rows[-1]['AP-small'])

In [ ]:
with (project/'hasil_m2/evaluasi/eksperimen_e2_val.csv').open() as stream:
    baseline = {r['id']: r for r in csv.DictReader(stream)}
for r in rows:
    r['perangkat'] = 'GPU' if device == '0' else 'CPU'
    r['ap_small_acuan'] = float(baseline[r['id']]['AP-small'])
    r['selisih_dari_acuan'] = r['AP-small'] - r['ap_small_acuan']
with (output/'eksperimen.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print('Perubahan AP-small 1280 terhadap 640:', rows[1]['AP-small'] - rows[0]['AP-small'])
print('CSV dan prediksi per resolusi tersedia di Output.')

run = {"stage": "E2", "training_performed": False, "checkpoint_origin": "Historical native E2, five epochs",
       "inference_performed": True, "cpu_benchmark_performed": True, "python": platform.python_version(),
       "torch": torch.__version__, "ultralytics": version("ultralytics"), "pycocotools": version("pycocotools"),
       "cpu_affinity": affinity, "validation_images": 356, "validation_boxes": 2205,
       "checkpoints_sha256": hashes, "csv_sha256": digest(output / "eksperimen.csv")}
(output / "run.json").write_text(json.dumps(run, indent=2))
print(json.dumps(run, indent=2))
